# 02 - Brand Specification Generator

##  Objective
This notebook demonstrates how to transform a raw, unstructured brand brief into a structured **Brand Specification** using a Large Language Model (LLM).

The Brand Specification is the **foundation** of the entire BRANDORA pipeline. Every subsequent step (name generation, slogan creation, color palette selection, logo generation) depends on this structured data.

---

##  What This Notebook Does

1. **Loads** a sample brand brief from `test_briefs.json`
2. **Connects** to Groq API (using secure environment variables)
3. **Sends** the brief to an LLM with a carefully crafted prompt
4. **Forces** the LLM to output structured JSON (not free-form text)
5. **Validates** the output to ensure it matches the expected schema

---

##  Key Technical Decisions

### Why Groq?
- Free tier with generous rate limits
- Fast inference (Llama 3.3 70B runs in <1 second)
- Supports JSON mode for structured outputs

### Why JSON Mode?
Without JSON mode, the LLM might output:

In [26]:

!git clone https://github.com/maram-elaian/brandora.git

import os
os.chdir('/kaggle/working/brandora')

print(os.getcwd())

!ls -l data/

fatal: destination path 'brandora' already exists and is not an empty directory.
/kaggle/working/brandora
total 16
-rw-r--r-- 1 root root 15771 Sep 20 09:47 test_briefs.json


In [27]:
!pip install -q groq
from groq import Groq
client = Groq(api_key=groq_key)


In [28]:
models = client.models.list()
print(len(models.data))
for i in models.data:
    print(f"- {i.id}")

13
- canopylabs/orpheus-v1-english
- whisper-large-v3
- openai/gpt-oss-120b
- canopylabs/orpheus-arabic-saudi
- allam-2-7b
- qwen/qwen3.8-27b
- whisper-large-v3-turbo
- openai/gpt-oss-20b
- meta-llama/llama-prompt-guard-2-86m
- meta-llama/llama-prompt-guard-2-22m
- groq/compound-mini
- groq/compound
- openai/gpt-oss-safeguard-20b


In [29]:
import json

# فتح ملف الاختبار
with open('data/test_briefs.json', 'r', encoding='utf-8') as file:
    test_briefs = json.load(file)

# اختيار أول حالة فقط (BR001 - مقهى للطلاب)
sample_brief = test_briefs[0]

# عرض البيانات للتأكد
print("✅ تم تحميل البيانات بنجاح!")
print(f"\n📝 حالة الاختبار المختارة:")
print(f"Industry: {sample_brief['industry']}")
print(f"Target Audience: {sample_brief['target_audience']}")
print(f"Purpose: {sample_brief['brand_purpose']}")

✅ تم تحميل البيانات بنجاح!

📝 حالة الاختبار المختارة:
Industry: coffee
Target Audience: university students
Purpose: provide a comfortable and energetic place for studying and socializing


#  Comparing 3 models 
---
   - qwen/qwen3.8-27b
   -  openai/gpt-oss-20b
   -  openai/gpt-oss-120b

In [33]:
system_prompt = """You are an expert Brand Strategist. 
Your task is to analyze a raw brand brief and extract a structured Brand Specification.
You MUST output ONLY valid JSON. Do not include any markdown formatting (like ```json) or extra text."""


user_prompt = f"""Analyze this brief:
Industry: {sample_brief['industry']}
Target Audience: {sample_brief['target_audience']}
Description: {sample_brief['brand_purpose']}

Return a JSON object with EXACTLY this structure:
{{
  "industry": "string",
  "target_audience": "string",
  "brand_purpose": "string",
  "personality": ["string", "string"],
  "tone": "string",
  "visual_style": ["string", "string"],
  "keywords": ["string", "string"],
  "color_direction": ["string", "string"],
  "logo_direction": "string"
}}"""

print("done")

done


In [32]:
#GPT-OSS-20B
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",  # اسم النموذج الدقيق من القائمة
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.2,
    response_format={"type": "json_object"}
)
raw_output = response.choices[0].message.content
clean_json_string = raw_output.replace("```json", "").replace("```", "").strip()
brand_spec = json.loads(clean_json_string)

print("✅ نجح GPT-OSS-20B! إليك النتيجة:")
print(json.dumps(brand_spec, indent=2, ensure_ascii=False))

✅ نجح GPT-OSS-20B! إليك النتيجة:
{
  "industry": "coffee",
  "target_audience": "university students",
  "brand_purpose": "provide a comfortable and energetic place for studying and socializing",
  "personality": [
    "Inviting",
    "Vibrant"
  ],
  "tone": "warm, upbeat, supportive",
  "visual_style": [
    "Modern minimal",
    "Cozy warm lighting"
  ],
  "keywords": [
    "study hub",
    "community vibe"
  ],
  "color_direction": [
    "warm neutrals",
    "bold accent"
  ],
  "logo_direction": "simple icon-based design featuring a coffee cup integrated with a book"
}


In [34]:
#gpt-oss-120b

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.2,
    response_format={"type": "json_object"}
)

raw_output = response.choices[0].message.content
clean_json = raw_output.replace("```json", "").replace("```", "").strip()
result_2 = json.loads(clean_json)

print("✅ النتيجة من openai/gpt-oss-120b:")
print(json.dumps(result_2, indent=2, ensure_ascii=False))

✅ النتيجة من openai/gpt-oss-120b:
{
  "industry": "coffee",
  "target_audience": "university students",
  "brand_purpose": "To create an energetic, comfortable space where students can study, collaborate, and recharge over quality coffee.",
  "personality": [
    "energetic",
    "welcoming"
  ],
  "tone": "Friendly and motivating",
  "visual_style": [
    "modern",
    "cozy"
  ],
  "keywords": [
    "study",
    "community"
  ],
  "color_direction": [
    "warm earth tones",
    "vibrant accent colors"
  ],
  "logo_direction": "A simple, clean mark combining a coffee cup with a subtle book or study element"
}


In [35]:
#qwen3.8-27b

response = client.chat.completions.create(
    model="qwen/qwen3.8-27b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.2,
    response_format={"type": "json_object"}
)

raw_output = response.choices[0].message.content
clean_json = raw_output.replace("```json", "").replace("```", "").strip()
result_3 = json.loads(clean_json)

print("✅ النتيجة من qwen/qwen3.8-27b:")
print(json.dumps(result_3, indent=2, ensure_ascii=False))

✅ النتيجة من qwen/qwen3.8-27b:
{
  "industry": "coffee",
  "target_audience": "university students",
  "brand_purpose": "To provide a comfortable and energetic environment that fuels academic focus and social connection for university students.",
  "personality": [
    "Energetic",
    "Welcoming"
  ],
  "tone": "Friendly and inspiring",
  "visual_style": [
    "Modern minimalism",
    "Student-centric"
  ],
  "keywords": [
    "Focus",
    "Community"
  ],
  "color_direction": [
    "Warm earth tones",
    "Vibrant accents"
  ],
  "logo_direction": "A clean, modern mark that subtly integrates elements of a coffee cup and a book or graduation cap to symbolize the blend of energy and study."
}
